<a href="https://colab.research.google.com/github/phamvanhung33182005-ux/Parameter_Identification_of_PMSM_using_PINN/blob/main/notebooks/03_baseline_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== CELL 1: LOAD DATASET VÀ KIỂM TRA GPU =====

from google.colab import drive
import os
import numpy as np
import torch

# Gắn Google Drive vào Google Colab.
drive.mount('/content/drive')

# Đường dẫn thư mục gốc của project.
PROJECT_DIR = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Đường dẫn tới dataset đã preprocessing.
PROCESSED_DIR = os.path.join(
    PROJECT_DIR,
    'dataset',
    'processed'
)

# Kiểm tra thư mục dataset có tồn tại hay không.
print('Processed dataset directory:', PROCESSED_DIR)
print('Directory exists:', os.path.exists(PROCESSED_DIR))

# Đọc input training.
X_train = np.load(
    os.path.join(PROCESSED_DIR, 'X_train.npy')
)

# Đọc target training.
Y_train = np.load(
    os.path.join(PROCESSED_DIR, 'Y_train.npy')
)

# Đọc input validation.
X_validation = np.load(
    os.path.join(PROCESSED_DIR, 'X_validation.npy')
)

# Đọc target validation.
Y_validation = np.load(
    os.path.join(PROCESSED_DIR, 'Y_validation.npy')
)

# Đọc input test.
X_test = np.load(
    os.path.join(PROCESSED_DIR, 'X_test.npy')
)

# Đọc target test.
Y_test = np.load(
    os.path.join(PROCESSED_DIR, 'Y_test.npy')
)

# In kích thước dataset.
print('\n===== DATASET SHAPE =====')

print('X_train      :', X_train.shape)
print('Y_train      :', Y_train.shape)

print('X_validation :', X_validation.shape)
print('Y_validation :', Y_validation.shape)

print('X_test       :', X_test.shape)
print('Y_test       :', Y_test.shape)

# Kiểm tra phiên bản PyTorch.
print('\n===== PYTORCH / GPU =====')

print('PyTorch version:', torch.__version__)

# Kiểm tra CUDA.
cuda_available = torch.cuda.is_available()

print('CUDA available:', cuda_available)

# Nếu có GPU thì in tên GPU.
if cuda_available:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU: Không khả dụng.')

# Tự động chọn GPU nếu có CUDA.
device = torch.device(
    'cuda' if cuda_available else 'cpu'
)

print('Selected device:', device)

Mounted at /content/drive
Processed dataset directory: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN/dataset/processed
Directory exists: True

===== DATASET SHAPE =====
X_train      : (90000, 5)
Y_train      : (90000, 3)
X_validation : (30000, 5)
Y_validation : (30000, 3)
X_test       : (30000, 5)
Y_test       : (30000, 3)

===== PYTORCH / GPU =====
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Selected device: cuda


In [2]:
# ===== CELL 2: KIỂM TRA INPUT / OUTPUT VÀ TẠO TENSOR =====

# Khai báo tên các input features.
INPUT_FEATURES = [
    'ud',
    'uq',
    'id',
    'iq',
    'nm'
]

# Khai báo tên các output targets.
TARGET_FEATURES = [
    'Rs',
    'Ld',
    'Lq'
]

# In thông tin input.
print('===== INPUT FEATURES =====')

print('Input features:', INPUT_FEATURES)

print('Number of input features:', len(INPUT_FEATURES))

# In thông tin output.
print('\n===== OUTPUT TARGETS =====')

print('Target features:', TARGET_FEATURES)

print('Number of target features:', len(TARGET_FEATURES))

# Kiểm tra số chiều thực tế của dataset.
assert X_train.shape[1] == len(INPUT_FEATURES), \
    'Số input features không khớp.'

assert Y_train.shape[1] == len(TARGET_FEATURES), \
    'Số output targets không khớp.'

# Chuyển NumPy array thành PyTorch Tensor.
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

Y_train_tensor = torch.tensor(
    Y_train,
    dtype=torch.float32
)

X_validation_tensor = torch.tensor(
    X_validation,
    dtype=torch.float32
)

Y_validation_tensor = torch.tensor(
    Y_validation,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

Y_test_tensor = torch.tensor(
    Y_test,
    dtype=torch.float32
)

# Kiểm tra shape sau khi chuyển sang Tensor.
print('\n===== TENSOR SHAPE =====')

print('X_train_tensor      :', X_train_tensor.shape)
print('Y_train_tensor      :', Y_train_tensor.shape)

print('X_validation_tensor :', X_validation_tensor.shape)
print('Y_validation_tensor :', Y_validation_tensor.shape)

print('X_test_tensor       :', X_test_tensor.shape)
print('Y_test_tensor       :', Y_test_tensor.shape)

# Kiểm tra dtype.
print('\nTensor dtype:', X_train_tensor.dtype)

# Hiển thị một sample.
print('\n===== SAMPLE =====')

print('Input sample :', X_train_tensor[0])
print('Target sample:', Y_train_tensor[0])

===== INPUT FEATURES =====
Input features: ['ud', 'uq', 'id', 'iq', 'nm']
Number of input features: 5

===== OUTPUT TARGETS =====
Target features: ['Rs', 'Ld', 'Lq']
Number of target features: 3

===== TENSOR SHAPE =====
X_train_tensor      : torch.Size([90000, 5])
Y_train_tensor      : torch.Size([90000, 3])
X_validation_tensor : torch.Size([30000, 5])
Y_validation_tensor : torch.Size([30000, 3])
X_test_tensor       : torch.Size([30000, 5])
Y_test_tensor       : torch.Size([30000, 3])

Tensor dtype: torch.float32

===== SAMPLE =====
Input sample : tensor([ 1.2205, -2.2199,  0.5614, -1.2761, -1.6356])
Target sample: tensor([-1.4635, -0.5408,  0.4599])


In [4]:
# ===== CELL 3: XÂY DỰNG BASELINE MLP =====

import torch.nn as nn

# Định nghĩa class BaselineMLP.
class BaselineMLP(nn.Module):

    # Hàm khởi tạo model.
    def __init__(self):

        # Gọi constructor của nn.Module.
        super().__init__()

        # Xây dựng kiến trúc mạng.
        self.network = nn.Sequential(

            # Input layer:
            # 5 input features -> 64 neurons.
            nn.Linear(5, 64),

            # Activation function.
            nn.ReLU(),

            # Hidden layer:
            # 64 neurons -> 64 neurons.
            nn.Linear(64, 64),

            # Activation function.
            nn.ReLU(),

            # Output layer:
            # 64 neurons -> 3 outputs.
            nn.Linear(64, 3)
        )

    # Định nghĩa forward propagation.
    def forward(self, x):

        # Đưa input đi qua toàn bộ network.
        return self.network(x)


# Khởi tạo Baseline MLP.
model = BaselineMLP()

# Đưa model lên device đã chọn.
model = model.to(device)

# In kiến trúc model.
print(model)

BaselineMLP(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)


In [5]:
# ===== CELL 4: KIỂM TRA PARAMETERS =====

# Tính tổng số parameters.
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

# Tính số trainable parameters.
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

# In kết quả.
print('===== MODEL PARAMETERS =====')

print('Total parameters    :', total_parameters)

print('Trainable parameters:', trainable_parameters)

===== MODEL PARAMETERS =====
Total parameters    : 4739
Trainable parameters: 4739


In [6]:
# ===== CELL 5: KIỂM TRA FORWARD PASS =====

# Đưa model về evaluation mode.
model.eval()

# Lấy 32 sample đầu tiên.
sample_input = X_train_tensor[:32]

# Đưa sample lên GPU hoặc CPU.
sample_input = sample_input.to(device)

# Tắt gradient vì đây chỉ là kiểm tra forward pass.
with torch.no_grad():

    # Cho input đi qua MLP.
    sample_output = model(sample_input)

# In shape.
print('===== FORWARD PASS =====')

print('Input shape :', sample_input.shape)

print('Output shape:', sample_output.shape)

# In một vài output.
print('\nOutput sample:')

print(sample_output[:5])

===== FORWARD PASS =====
Input shape : torch.Size([32, 5])
Output shape: torch.Size([32, 3])

Output sample:
tensor([[ 0.2647, -0.0262, -0.0265],
        [ 0.2643, -0.0261, -0.0263],
        [ 0.2629, -0.0260, -0.0255],
        [ 0.2606, -0.0258, -0.0243],
        [ 0.2575, -0.0255, -0.0226]], device='cuda:0')


In [7]:
# ===== CELL 6: KIỂM TRA DEVICE CỦA MODEL =====

# Lấy parameter đầu tiên của model.
first_parameter = next(model.parameters())

# Kiểm tra device của parameter.
model_device = first_parameter.device

# In kết quả.
print('Model device   :', model_device)

print('Selected device:', device)

# Kiểm tra model và device có khớp hay không.
if model_device.type == device.type:

    print('Device check: PASS')

else:

    print('Device check: FAIL')

Model device   : cuda:0
Selected device: cuda
Device check: PASS


In [8]:
# ===== CELL 7: BASELINE SUMMARY =====

print('========== BASELINE MLP SUMMARY ==========')

print('Input features :', INPUT_FEATURES)

print('Output targets :', TARGET_FEATURES)

print('Architecture   : 5 -> 64 -> 64 -> 3')

print('Activation     : ReLU')

print('Total params   :', total_parameters)

print('Device         :', model_device)

print('\nTraining       : NOT PERFORMED')

print('Tuning         : NOT PERFORMED')

print('Status         : Baseline model ready for training')

========== BASELINE MLP SUMMARY ==========
Input features : ['ud', 'uq', 'id', 'iq', 'nm']
Output targets : ['Rs', 'Ld', 'Lq']
Architecture   : 5 -> 64 -> 64 -> 3
Activation     : ReLU
Total params   : 4739
Device         : cuda:0

Training       : NOT PERFORMED
Tuning         : NOT PERFORMED
Status         : Baseline model ready for training
